# RSI Mean Reversion — Oversold Bounce Strategy
Buy when RSI(14) < 30, sell when RSI > 70. Testing across multiple tickers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use('dark_background')

In [ ]:
def compute_rsi(prices, period=14):
    delta = prices.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def backtest_rsi(ticker, start='2019-01-01', end='2024-12-31', rsi_buy=30, rsi_sell=70):
    df = yf.download(ticker, start=start, end=end, progress=False)
    df['rsi'] = compute_rsi(df['Close'])
    
    capital = 100_000
    cash = capital
    shares = 0
    trades = []
    
    for i in range(14, len(df)):
        rsi = df['rsi'].iloc[i]
        price = df['Close'].iloc[i]
        
        if rsi < rsi_buy and shares == 0:
            shares = int(cash * 0.95 / price)
            entry = price
            cash -= shares * price
        elif rsi > rsi_sell and shares > 0:
            pnl = (price - entry) * shares
            cash += shares * price
            trades.append({'pnl': pnl, 'ret': (price/entry - 1) * 100})
            shares = 0
    
    final = cash + shares * df['Close'].iloc[-1]
    total_ret = final / capital - 1
    win_rate = sum(1 for t in trades if t['pnl'] > 0) / max(len(trades), 1) * 100
    
    return {
        'ticker': ticker,
        'total_return': f'{total_ret:.1%}',
        'trades': len(trades),
        'win_rate': f'{win_rate:.0f}%',
        'avg_trade': f'{np.mean([t["ret"] for t in trades]):.1f}%' if trades else 'N/A'
    }

In [ ]:
# Test across multiple tickers
tickers = ['AAPL', 'MSFT', 'GOOGL', 'SPY', 'QQQ', 'TSLA', 'AMZN', 'NVDA']
results = [backtest_rsi(t) for t in tickers]
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# RSI threshold sensitivity analysis
thresholds = [(20, 80), (25, 75), (30, 70), (35, 65)]
sensitivity = []

for buy_t, sell_t in thresholds:
    r = backtest_rsi('SPY', rsi_buy=buy_t, rsi_sell=sell_t)
    r['thresholds'] = f'{buy_t}/{sell_t}'
    sensitivity.append(r)

print('\nRSI Threshold Sensitivity (SPY):')
print(pd.DataFrame(sensitivity)[['thresholds', 'total_return', 'trades', 'win_rate']].to_string(index=False))